In [ ]:
!pip install rdkit
!pip install pandas
!pip install numpy



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [3]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize

In [4]:
df = pd.read_csv("https://huggingface.co/datasets/scikit-fingerprints/MoleculeNet_HIV/resolve/main/hiv.csv")
df.head()

,SMILES,label
0,CCC1=[O+][Cu-3]2([O+]=C(CC)C1)[O+]=C(CC)CC(CC)...,0
1,C(=Cc1ccccc1)C1=[O+][Cu-3]2([O+]=C(C=Cc3ccccc3...,0
2,CC(=O)N1c2ccccc2Sc2c1ccc1ccccc21,0
3,Nc1ccc(C=Cc2ccc(N)cc2S(=O)(=O)O)c(S(=O)(=O)O)c1,0
4,O=S(=O)(O)CCS(=O)(=O)O,0


In [5]:
display(df.SMILES.isna().sum())
display(df.label.isna().sum())

np.int64(0)

np.int64(0)

In [6]:
display(df.SMILES.isin(['None', 'none', " ", 'na']).sum())
display(df.label.isin(['None', 'none', " ", 'na']).sum())

np.int64(0)

np.int64(0)

In [7]:
clean_smiles, fps, metrics, labels = [], [], [], []
invalid = 0

In [8]:
chooser = rdMolStandardize.LargestFragmentChooser()
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

In [9]:
for smi, y in zip(df["SMILES"], df["label"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        invalid += 1
        continue
    mol = chooser.choose(mol)
    clean_smiles.append(Chem.MolToSmiles(mol))
    fps.append(gen.GetFingerprintAsNumPy(mol))
    metrics.append(Descriptors.CalcMolDescriptors(mol))
    labels.append(y)

print("Invalid SMILES removed:", invalid)

Invalid SMILES removed: 0


In [10]:
X_fp = pd.DataFrame(fps).add_prefix("fp_")
X_metrics = pd.DataFrame(metrics)
df = pd.concat([X_fp, X_metrics], axis=1)
df["smiles"] = clean_smiles
df["label"] = labels

df = df.drop_duplicates(subset="smiles")     # remove duplicates after canonicalizing
print(df.shape, df["label"].mean()*100)

(40959, 1243) 3.4644400498059036


In [11]:
df.head(5)


,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,smiles,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,CCC1=[O+][Cu-3]2([O+]=C(CC)C1)[O+]=C(CC)CC(CC)...,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,C(=Cc1ccccc1)C1=[O+][Cu-3]2([O+]=C(C=Cc3ccccc3...,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,CC(=O)N1c2ccccc2Sc2c1ccc1ccccc21,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Nc1ccc(C=Cc2ccc(N)cc2S(=O)(=O)O)c(S(=O)(=O)O)c1,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,O=S(=O)(O)CCS(=O)(=O)O,0


In [12]:
df[X_metrics.columns]

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,5.925903,5.925903,0.759994,-2.330012,0.434358,23.368421,319.888,295.696,319.096508,114,...,0,0,0,0,0,0,0,0,0,0
1,6.515117,6.515117,0.475558,-2.879653,0.104103,20.717949,564.140,535.916,563.127808,198,...,0,0,0,0,0,0,0,0,0,0
2,12.177015,12.177015,0.041001,0.041001,0.581359,12.904762,291.375,278.271,291.071785,102,...,0,0,0,0,0,0,0,0,0,0
3,11.401803,11.401803,0.087790,-4.524487,0.357615,12.583333,370.408,356.296,370.029328,128,...,0,0,0,0,0,0,0,0,0,0
4,9.857917,9.857917,0.980324,-4.296481,0.538723,13.400000,190.198,184.150,189.960580,62,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41122,13.370142,13.370142,0.007185,-0.366239,0.804191,32.035714,379.504,350.272,379.225977,148,...,0,0,0,0,0,0,0,0,0,0
41123,14.160262,14.160262,0.075082,-0.480928,0.427745,28.888889,484.596,452.340,484.236208,186,...,0,0,0,0,0,0,0,0,0,0
41124,13.962959,13.962959,0.011372,-0.415582,0.450949,29.212121,440.587,408.331,440.246378,170,...,0,0,0,0,0,0,0,0,0,0
41125,13.989331,13.989331,0.009754,-0.416314,0.450949,29.212121,440.587,408.331,440.246378,170,...,0,0,0,0,0,0,0,0,0,0


In [13]:
df[X_fp.columns]

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,fp_1014,fp_1015,fp_1016,fp_1017,fp_1018,fp_1019,fp_1020,fp_1021,fp_1022,fp_1023
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41122,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
41123,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
41124,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
41125,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [14]:
raw = pd.DataFrame({"smiles": clean_smiles, "label": labels})
conflicts = raw.groupby("smiles")["label"].nunique()
conflicts = conflicts[conflicts > 1].index
print("Conflicts:", len(conflicts))

Conflicts: 16


In [16]:
df = df[~df["smiles"].isin(conflicts)]
df = df.drop_duplicates(subset="smiles")

In [17]:
df = df.replace([np.inf, -np.inf], np.nan)
df = df.drop(columns=[c for c in X_metrics.columns if df[c].nunique() <= 1])  # constant columns
print("NaN/inf cells before fill:", df.drop(columns="smiles").replace([np.inf, -np.inf], np.nan).isna().sum().sum())
df = df.fillna(df.median(numeric_only=True))
print("NaN left:", df.drop(columns="smiles").isna().sum().sum())

NaN/inf cells before fill: 13944
NaN left: 0


In [ ]:
num_cols = [c for c in X_metrics.columns if c in df.columns]
df[num_cols] = df[num_cols].clip(df[num_cols].quantile(0.01), df[num_cols].quantile(0.99), axis=1)

In [ ]:
!pip install scikit-learn
from sklearn.model_selection import train_test_split

X = df.drop(columns=["smiles", "label"])
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(X_train.shape, X_test.shape)


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\ACC\AppData\Local\Temp\ipykernel_30904\3999664994.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train.assign(label=y_train).to_csv("train.csv", index=False)
C:\Users\ACC\AppData\Local\Temp\ipykernel_30904\3999664994.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test.assign(label=y_test).to_csv("test.csv", index=False)


(32754, 1239) (8189, 1239)
